# Demo (Modular, Deterministic, No Normalizer)

Minimal, testable steps:
1) Patch `PYTHONPATH` and create/check experiment directories
2) Clone `phenopacket-store`, build PMID list, fetch PDFs, build dataset CSV
3) Load dataset, validate columns, dedupe by PMID
4) **PDF -> Text verification** (preview + full dumps to disk) <- _inspect conversion here_
5) Align ground-truth phenopackets
6) Sanity inference (single case) -- model must output exact HPO `id` + primary `label`
7) Batch inference across aligned cases
8) Evaluation (HPO-only) vs. ground truth

**Important:** There is **no** post-hoc HPO normalization or ontology lookup. The model must produce the exact `HP:#######` id and the official primary label by itself. We use `temperature=0` and structured JSON output to reduce drift.

In [ ]:
# Step 1: Pathing + experiment directories

# Add project root (parent of this notebooks/ folder) to sys.path
import sys, os, logging
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


# Configure concise, notebook-friendly logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", stream=sys.stdout)
logger = logging.getLogger("p5-notebook")
logger.setLevel(logging.INFO)

from notebooks.utils.demonstration_directory_creation import (
   patch_pythonpath_and_create_demonstration_directories,
)

# Adds src/ and notebooks/utils/ to sys.path, creates experiment folders, and validates imports
paths = patch_pythonpath_and_create_demonstration_directories(emit_verbose_logs=True)

# Export human-friendly names used throughout the notebook
project_root = str(paths.project_root_directory)
src_folder = str(paths.source_code_directory)
utils_folder = str(paths.notebooks_utilities_directory)
pdf_input_directory = str(paths.pdf_input_directory)
ground_truth_notebooks_directory = str(paths.ground_truth_notebooks_directory)
dataset_csv_path = str(paths.dataset_csv_file_path)
experimental_data_root = str(paths.experimental_data_root_directory)
llm_output_directory = str(paths.llm_raw_output_directory)
validated_jsons_directory = str(paths.validated_jsons_output_directory)
evaluation_report_output_path = str(paths.evaluation_report_file_path)

logger.info("[OK] Step 1 complete: directories + PYTHONPATH are set up.")

In [ ]:
# Step 2: phenopacket-store bootstrap (Stages 1-4)

from notebooks.utils.phenopacket_store_dataset_setup import setup_phenopacket_store_dataset

# Choose how many PDFs to download:
#   0 = all available; N = first N (useful for quick smoke tests)
maximum_pdf_download_count_for_setup = 10

pmid_pickle_file_path = setup_phenopacket_store_dataset(
   src_folder=src_folder,
   pdf_input_directory=pdf_input_directory,
   ground_truth_notebooks_directory=ground_truth_notebooks_directory,
   dataset_csv_path=dataset_csv_path,
   max_pdfs_to_download=maximum_pdf_download_count_for_setup,
)

logger.info("[OK] Step 2 complete: phenopacket-store bootstrapped.")
logger.info("pmid_pickle_file_path: %s", pmid_pickle_file_path)
logger.info("dataset_csv_path: %s", dataset_csv_path)

In [ ]:
# Step 3: Load dataset CSV, validate columns, dedupe

from IPython.display import display
from notebooks.utils.dataset_loading import load_and_validate_dataset

# How many "input" paths to quickly existence-check (set 0 to skip)
max_input_existence_checks = 10

dataframe_cases, dataset_stats = load_and_validate_dataset(
   dataset_csv_path=dataset_csv_path,
   max_input_existence_checks=max_input_existence_checks,
   verbose=True,   # logs via logger inside the util
)

logger.info("[dataset] Stats: %s", dataset_stats)

# Optional: small preview in the notebook UI
display(dataframe_cases.head(5))

In [ ]:
# Step 4: PDF -> text cache, preview & full dumps for verification

from pathlib import Path
from notebooks.utils.pdf_text_cache import PdfTextCache

# Use the API implemented in notebooks/utils/pdf_text_cache.py
pdf_text_cache = PdfTextCache(experimental_data_root=experimental_data_root)

# Where to dump full extracted text for manual inspection
debug_dump_directory = Path(experimental_data_root) / "text_cache" / "debug_dumps"
debug_dump_directory.mkdir(parents=True, exist_ok=True)

# Convert a subset (or all) to verify pipeline correctness
num_smoke_tests = len(dataframe_cases)
logger.info("[text] Converting first %d document(s) to text...", num_smoke_tests)

for idx, input_file_path in enumerate(dataframe_cases["input"].head(num_smoke_tests), start=1):
   try:
       extracted_text = pdf_text_cache.get_text(str(input_file_path))  # <-- current API
       collapsed = " ".join(extracted_text.split())
       preview_length = 1200
       logger.info("  [%d] %s -> %d characters", idx, input_file_path, len(extracted_text))
       logger.info("      preview (first %d chars): %s...", preview_length, collapsed[:preview_length])

       base_name = Path(str(input_file_path)).stem
       debug_text_path = debug_dump_directory / f"{base_name}.txt"
       debug_text_path.write_text(extracted_text, encoding="utf-8")
       logger.info("      saved full text -> %s", debug_text_path)
   except Exception as e:
       logger.warning("  [%d] %s -> ERROR: %s", idx, input_file_path, e)

In [ ]:
# Step 5: Load & align ground-truth phenopackets with the dataset

from notebooks.utils.truth_alignment import load_and_align_truth

# Limit rows for a quick pass; set to None to process all rows
maximum_rows_to_align = None

truth_alignment = load_and_align_truth(
   dataset_dataframe=dataframe_cases,
   maximum_rows=maximum_rows_to_align,
)

pmids_aligned = truth_alignment.list_of_pmids_aligned
truth_packets_wrapped = truth_alignment.list_of_truth_packets_wrapped
patient_ids_from_truth = truth_alignment.list_of_patient_ids_from_truth
input_paths_aligned = truth_alignment.list_of_input_paths_aligned
skipped_truth_cases = truth_alignment.list_of_skipped_cases

logger.info("[truth] aligned cases: %d", len(pmids_aligned))
if skipped_truth_cases:
   logger.info("[truth] skipped cases: %d (showing up to 5)", len(skipped_truth_cases))
   for item in skipped_truth_cases[:5]:
       logger.info("   - %s", item)

# Optional: quick peek at first packet's phenotypes
if truth_packets_wrapped:
   try:
       first_pheno_preview = truth_packets_wrapped[0].list_phenotypes()
       logger.info("[truth] first packet phenotype count: %d", len(first_pheno_preview))
   except Exception as e:
       logger.warning("[truth] preview error: %s", e)

In [ ]:
# Step 6: Sanity inference on a single aligned case

import json
from notebooks.utils.hpo_extraction import extract_hpo_terms, build_minimal_phenopacket_from_hpo_list
from phenopacket import Phenopacket as UtilPhenopacket

if not pmids_aligned:
   raise RuntimeError("No aligned cases available. Ensure the truth-alignment step succeeded.")

# Choose the first aligned case for a quick smoke test
example_index = 0
example_pmid = pmids_aligned[example_index]
example_patient_id = patient_ids_from_truth[example_index]
example_input_path = input_paths_aligned[example_index]

logger.info("[inference] Using PMID=%s | patient_id=%s", example_pmid, example_patient_id)
logger.info("[inference] Source file: %s", example_input_path)

# Load cached text version
clinical_text_for_example = pdf_text_cache.get_text(str(example_input_path))
logger.info("[inference] Loaded %d characters of clinical text", len(clinical_text_for_example))

# Deterministic extraction with chunking and strict JSON validation
predicted_terms_for_example, raw_model_text = extract_hpo_terms(
  clinical_text=clinical_text_for_example,
  model="llama3.2:latest",
  return_raw_model_text=True,
  debug_logging=True,
  max_pheno_items=20,
  timeout_s=60,             # per-chunk timeout
  chunk_max_chars=3000,     # safe chunk size
  chunk_overlap_chars=300,  # overlap for context
)

logger.info("[inference] extracted %d HPO term object(s)", len(predicted_terms_for_example))
logger.info("[inference] preview: %s", json.dumps(predicted_terms_for_example[:5], indent=2))

# Build a minimal phenopacket (no normalization/lookup)
predicted_packet_json = build_minimal_phenopacket_from_hpo_list(
  patient_id=example_patient_id,
  hpo_list=predicted_terms_for_example,
)
predicted_packet_util = UtilPhenopacket(predicted_packet_json)
logger.info("[inference] phenotypicFeatures count: %d", len(predicted_packet_util.list_phenotypes()))

# Persist raw model output for auditing
from pathlib import Path
Path(llm_output_directory).mkdir(parents=True, exist_ok=True)
raw_dump_path = Path(llm_output_directory) / f"{example_pmid}__raw.txt"
raw_dump_path.write_text(raw_model_text or "", encoding="utf-8")
logger.info("[inference] saved raw model output -> %s", raw_dump_path)

In [ ]:
# Step 7: Batch inference across all aligned cases

from notebooks.utils.hpo_batch import run_hpo_batch_inference

aligned_pmids = truth_alignment.list_of_pmids_aligned
aligned_input_paths = truth_alignment.list_of_input_paths_aligned
aligned_patient_ids = truth_alignment.list_of_patient_ids_from_truth

logger.info("[batch] total aligned cases: %d", len(aligned_pmids))

batch_result = run_hpo_batch_inference(
   list_of_pmids_aligned=aligned_pmids,
   list_of_input_paths_aligned=aligned_input_paths,
   list_of_patient_ids_aligned=aligned_patient_ids,
   directory_for_raw_llm_outputs=llm_output_directory,
   directory_for_predicted_jsons=validated_jsons_directory,
   ollama_model_name="llama3.2:latest",
   sleep_seconds_between_cases=0.0,
   write_raw_model_text=True,
   write_predicted_json=True,
   debug_logging=False,
   timeout_s=60,   # shorter timeout than default
)

logger.info("[batch] successes = %d | failures = %d",
           batch_result.total_successes(), batch_result.total_failures())

# Store results for evaluation
predicted_packet_utils_for_evaluation = [
   o.predicted_packet_util for o in batch_result.successful_outcomes
]

# Peek at a couple of outputs
for outcome in batch_result.successful_outcomes[:3]:
   logger.info("  ? PMID %s: raw=%s | json=%s",
               outcome.pmid, outcome.raw_output_path, outcome.predicted_json_path)

if batch_result.failed_outcomes:
   logger.warning("[batch] Failures (first 5 shown):")
   for outcome in batch_result.failed_outcomes[:5]:
       logger.warning("  ? PMID %s: %s", outcome.pmid, outcome.error_message)

In [ ]:
# Step 8: Evaluation (predictions vs ground truth, HPO-only)

from evaluation import PhenotypeEvaluator
from report import Report

if not predicted_packet_utils_for_evaluation:
   raise RuntimeError("No predicted packets available for evaluation. Run the batch step first.")

# Map pmid -> predicted packet (only for successful outcomes)
predicted_by_pmid = {
   outcome.pmid: outcome.predicted_packet_util
   for outcome in (getattr(globals().get('batch_result', None), 'successful_outcomes', []) or [])
   if outcome.predicted_packet_util is not None
}

truth_packets_for_eval = []
predicted_packets_for_eval = []

for pmid, truth_pp in zip(aligned_pmids, truth_alignment.list_of_truth_packets_wrapped):
   if pmid in predicted_by_pmid:
       truth_packets_for_eval.append(truth_pp)
       predicted_packets_for_eval.append(predicted_by_pmid[pmid])

logger.info("[eval] evaluating %d predicted packets", len(predicted_packets_for_eval))

evaluator = PhenotypeEvaluator()
for predicted_pp, truth_pp in zip(predicted_packets_for_eval, truth_packets_for_eval):
   evaluator.check_phenotypes(predicted_pp.list_phenotypes(), truth_pp)

final_report = evaluator.report(
   creator="P5-demo-notebook",
   experiment="LLM HPO Extraction (HPO-only eval vs phenopacket-store)",
   model="ollama:llama3.2:latest",
   notes=f"Total evaluated pairs: {len(predicted_packets_for_eval)}",
)

# Try printing a friendly summary, fallback to raw contents if necessary
try:
   logger.info("=== Evaluation Summary ===\n%s", final_report.get_summary())
except Exception:
   logger.info("=== Evaluation Summary (raw) ===")
   logger.info("%s", getattr(final_report, "metrics", "<no metrics>"))
   logger.info("%s", getattr(final_report, "metadata", "<no metadata>"))